## Main inter-annotator agreement

This section evaluates KA and ACC metrics using only the Main1 and Main2 annotation phases. Pilot annotations are excluded, as they served primarily as training for the annotators. Additionally, the pilot stages involved three annotators, whereas the main annotation phases involved only two.

In [1]:
import pandas as pd
import krippendorff
import glob
import os

In [3]:
main_data = glob.glob('../Annotations/main*.csv')

dfs = [pd.read_csv(file) for file in main_data]
df = pd.concat(dfs, ignore_index=True)
df

,id,text,timestamp_tamara,flagged_tamara,comment_tamara,tag_tamara,timestamp_katja,flagged_katja,comment_katja,tag_katja,rev_tamara,rev_katja,final_tag,disagreement6
0,501,"Vlada nas sili, da sprejmemo tri zakone, ne da...",2023-04-05 20:14:36,NaN,"Implied opinion, could be neutral, but for me ...",Negative,2023-04-05 08:56:32,NaN,"Statement, but negative, am between N and N_Ne...",N_Neutral,NaN,It is more negative than N_Neutral; change to ...,Negative,1
1,502,"Do danes, do današnjega dne nobena od trditev ...",2023-04-05 20:15:22,NaN,"No explicit opinion, so I'd go with neutral, b...",N_Neutral,2023-04-05 08:58:53,NaN,"Defense of somebody, would say it is still a s...",P_Neutral,"I agree with Katja, change to P_Neutral",NaN,P_Neutral,1
2,503,"Mislim, da mesec dni počakati s temi stvarmi, ...",2023-04-05 20:15:54,NaN,"Clear opinion, negative sentiment.",Negative,2023-04-05 09:01:13,1.0,"Makes almost zero sense, so def neutral, even ...",P_Neutral,NaN,"I agree with Tamara, this is mostly negative.",Negative,1
3,504,Vodja Severa na proslavi javno primerja orožje...,2023-04-05 20:16:26,NaN,That is a very negative sentence. No opinion t...,Negative,2023-04-05 09:01:34,NaN,NaN,Negative,NaN,NaN,Negative,0
4,505,"Še enkrat, če hočete, za nove davkoplačevalce.",2023-04-05 20:17:56,NaN,This could very well be N_Neutral because of t...,P_Neutral,2023-04-05 09:01:56,NaN,"Statement, nothing inherently negative.",P_Neutral,NaN,NaN,P_Neutral,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2095,2596,Ustavno sodišče je zavrglo pobudo državnega zb...,2023-06-15 23:58:35,NaN,statement,N_Neutral,2023-06-19 18:21:47,NaN,NaN,N_Neutral,NaN,NaN,N_Neutral,0
2096,2597,"Sama, kot predstavnica mlajše generacije, želi...",2023-06-15 23:59:02,NaN,"positive sentiment, but not string enough to b...",P_Neutral,2023-06-19 18:22:32,1.0,NaN,N_Neutral,NaN,I have no idea what I was thinking here.,P_Neutral,1
2097,2598,Kakor hitro gre za vprašanje izplačila odškodn...,2023-06-15 23:59:09,NaN,NaN,Negative,2023-06-19 18:22:38,NaN,NaN,Negative,NaN,NaN,Negative,0
2098,2599,"Če gledamo te velike nakupovalne centre, saj v...",2023-06-15 23:59:22,NaN,statement,P_Neutral,2023-06-19 18:22:53,NaN,NaN,P_Neutral,NaN,NaN,P_Neutral,0


In [4]:
tag_cols = ['tag_tamara', 'tag_katja']
tags = df[tag_cols].dropna()
tags = df[tag_cols]

In [5]:
unique_tag6 = pd.unique(df[tag_cols].values.ravel())
tag_to_code6 = {label: idx for idx, label in enumerate(unique_tag6)}

for col in tag_cols:
    df[col + '_6class'] = df[col].map(tag_to_code6)

data_6class = df[[col + '_6class' for col in tag_cols]].to_numpy().T
alpha_6class = krippendorff.alpha(reliability_data=data_6class, level_of_measurement='nominal')

In [6]:
print("KA 6-class score: ", alpha_6class)

KA 6-class score:  0.5121644319934712


In [6]:
mapping_3class = {
    'Positive': 'Positive',
    'M_Positive': 'Positive',    
    'Negative': 'Negative',
    'M_Negative': 'Negative',
    'P_Neutral': 'Neutral',
    'N_Neutral': 'Neutral',
    }

for col in tag_cols:
    df[col + '_3class'] = df[col].map(mapping_3class)

unique_tag3 = pd.unique(df[[col + "_3class" for col in tag_cols]].values.ravel())
tag_to_code3 = {label: idx for idx, label in enumerate(unique_tag3)}

for col in tag_cols:
    df[col + '_3code'] = df[col + '_3class'].map(tag_to_code3)

data_3class = df[[col + '_3code' for col in tag_cols]].to_numpy().T
alpha_3class = krippendorff.alpha(reliability_data=data_3class, level_of_measurement="nominal")

In [7]:
print("KA 3-class score: ", alpha_3class)

KA 3-class score:  0.5402908445318997


In [8]:
#----Percentage of agreed-upon instances---#
def percent_agreement(rows):
    values = [val for val in rows if pd.notnull(val)]
    return int(len(set(values)) == 1)

agreement_6 = df[tag_cols].apply(percent_agreement, axis=1).mean()
agreement_3 = df[[col + '_3class' for col in tag_cols]].apply(percent_agreement, axis=1).mean()

In [9]:
print("Agreement for a 6-class schema: ", (agreement_6 * 100))
print("Agreement for a 3-class schema: ", (agreement_3 * 100))

Agreement for a 6-class schema:  65.14285714285715
Agreement for a 3-class schema:  74.90476190476191


## Label distribution

In [10]:
df['final_tag'] = df['final_tag'].astype("str").str.strip().str.replace(r"\(S\)", "", regex=True)
mapping_mistakes = {
'Negative.' : 'Negative',
'Postive':'Positive',
'Negatvie': 'Negative'
}
df['final_tag'] = df['final_tag'].map(mapping_mistakes).fillna(df['final_tag'])
df['final_tag'].unique()


array(['Negative', 'P_Neutral', 'N_Neutral', 'Positive', 'M_Positive',
       'M_Negative', 'Negative '], dtype=object)

In [11]:
df['final_tag'].value_counts()

final_tag
Negative      750
P_Neutral     632
N_Neutral     546
Positive       87
M_Positive     32
M_Negative     27
Negative       26
Name: count, dtype: int64

In [12]:
df['final_tag3'] = df['final_tag'].map(mapping_3class)
df['final_tag3'].unique()

array(['Negative', 'Neutral', 'Positive', nan], dtype=object)

In [13]:
df['final_tag3'].value_counts()

final_tag3
Neutral     1178
Negative     777
Positive     119
Name: count, dtype: int64